# Multi-head causal attention with dropout

This notebook continues from `causal_attention.ipynb`. A single attention head
computes, for each token, **one** weighted average of the other tokens. A GPT
model uses **multi-head attention**: several heads in parallel, each with its
own query, key, and value projections, so that each head can learn to look
for a different kind of relation between tokens.

We build it in two ways:

1. **Conceptually**, as several `CausalAttention` heads whose outputs are
   concatenated.
2. **Efficiently**, as the `MultiHeadAttention` class used in GPT models,
   which computes all the heads at once with a few reshapes.

Then we check that the two give the same result. The example follows the
multi-head attention section in Chapter 3 of *Build a Large Language Model
(From Scratch)* by Sebastian Raschka.

> **Origin and license.** This notebook is derived from [`ch03/01_main-chapter-code/ch03.ipynb`](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb) in the repository [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Copyright 2023-2026 Sebastian Raschka, licensed under the [Apache License 2.0](../LICENSES/Apache-2.0-LLMs-from-scratch.txt).
>
> **Modified by L. Saetta (2026):** explanations rewritten and extended, and code adapted to this repository. See [NOTICE](../NOTICE).

In [1]:
import torch
from torch import nn

## 1. Why multiple heads?

In a sentence, a token is related to other tokens in many ways at once. In
"The cat that I saw yesterday was sleeping", the verb "was" relates to its
subject "cat", while "yesterday" relates to "saw". One head gives each token
a single distribution of attention weights, so it must mix all these
relations into one average.

With `num_heads` heads, each token gets `num_heads` independent weighted
averages, one per head. During training, each head can specialize: for
example, one may follow the subject of a verb, another the previous token.
These roles are not programmed: they emerge from training, and they are not
always easy to interpret.

Multi-head attention does not make the model larger. The output size `d_out`
is **split** across the heads: each head works with vectors of size
`head_dim = d_out / num_heads`. GPT-2 Small, for example, has `d_out = 768`
and 12 heads, so each head works with 64 dimensions. The outputs of the heads
are concatenated back to size `d_out`, and a final linear layer, the
**output projection**, combines them.

We use the same six input vectors and the same batch of two identical
sequences as in the previous notebook.

In [2]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],  # step     (x^6)
    ]
)

batch = torch.stack((inputs, inputs), dim=0)

# 2 sequences, with 6 tokens each, and 3 features per token.
print(batch.shape)

torch.Size([2, 6, 3])


## 2. Multi-head attention as several causal heads

The most direct way to build multi-head attention is to create several
`CausalAttention` modules, run each of them on the same input, and
concatenate their outputs along the last dimension.

`CausalAttention` is the class of `causal_attention.ipynb`, repeated here so
that this notebook is independent.

In [3]:
class CausalAttention(nn.Module):
    """Single-head causal self-attention with dropout, for batches of sequences.

    Args:
        d_in: Number of features in each input token embedding.
        d_out: Number of features in each query, key, value, and output vector.
        context_length: Maximum number of tokens in a sequence.
        dropout: Probability of dropping an attention weight during training.
        qkv_bias: Whether each projection includes a trainable bias.
    """

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        """Compute context vectors with shape (b, num_tokens, d_out)."""
        _, num_tokens, _ = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values


class MultiHeadAttentionWrapper(nn.Module):
    """Multi-head attention as a list of independent causal attention heads.

    Args:
        d_in: Number of features in each input token embedding.
        d_out: Number of output features of each head.
        context_length: Maximum number of tokens in a sequence.
        dropout: Probability of dropping an attention weight during training.
        num_heads: Number of heads.
        qkv_bias: Whether each projection includes a trainable bias.
    """

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # nn.ModuleList registers the heads, so that their parameters belong
        # to this module (a plain Python list would hide them from PyTorch).
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        """Concatenate the outputs of the heads: (b, num_tokens, num_heads * d_out)."""
        return torch.cat([head(x) for head in self.heads], dim=-1)

Note that here `d_out` is the output size of **each** head: with two heads of
size 1, the output has `2 × 1 = 2` features per token.

This version is easy to read, but it runs the heads one after the other, in
a Python loop, with three small projections per head.

In [4]:
torch.manual_seed(123)

context_length = batch.shape[1]
d_in = batch.shape[2]
wrapper = MultiHeadAttentionWrapper(d_in, 1, context_length, 0.0, num_heads=2)

wrapper_vecs = wrapper(batch)
print(wrapper_vecs)
print("wrapper_vecs.shape:", wrapper_vecs.shape)

tensor([[[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]],

        [[-0.5740,  0.2216],
         [-0.7320,  0.0155],
         [-0.7774, -0.0546],
         [-0.6979, -0.0817],
         [-0.6538, -0.0957],
         [-0.6424, -0.1065]]], grad_fn=<CatBackward0>)
wrapper_vecs.shape: torch.Size([2, 6, 2])


## 3. The efficient implementation

`MultiHeadAttention` computes the same thing without a loop over the heads.
The idea is to do **one** large projection for all the heads, then split its
output into heads by reshaping the tensor:

1. Project the input with one `nn.Linear(d_in, d_out)` for the queries, one
   for the keys, and one for the values. Each output has shape
   `(b, num_tokens, d_out)`.
2. **Split** the last dimension into heads with `.view`:
   `(b, num_tokens, d_out)` → `(b, num_tokens, num_heads, head_dim)`.
   This does not move any data: it only changes how the same numbers are
   indexed. The first `head_dim` features belong to head 0, the next
   `head_dim` to head 1, and so on.
3. **Move the head dimension forward** with `.transpose(1, 2)`:
   `(b, num_heads, num_tokens, head_dim)`. Now the heads behave like an extra
   batch dimension.
4. Compute masked attention exactly as before. Matrix multiplication with `@`
   works on the last two dimensions and treats all the leading ones as batch
   dimensions, so all the heads of all the sequences are computed in a single
   operation. The scores have shape `(b, num_heads, num_tokens, num_tokens)`.
5. **Combine the heads**: transpose back to
   `(b, num_tokens, num_heads, head_dim)` and merge the last two dimensions
   with `.view(b, num_tokens, d_out)`. This is the concatenation of the heads.
6. Apply the **output projection**, `nn.Linear(d_out, d_out)`, which mixes
   the information coming from the different heads.

Two details in the code:

* `assert d_out % num_heads == 0`: the output size must split evenly across
  the heads.
* `.contiguous()` before the last `.view`: after a transpose, the numbers are
  no longer stored in memory in the order that `.view` needs, and
  `.contiguous()` makes a copy in the right order. `.reshape` would do the
  same automatically.

In [5]:
class MultiHeadAttention(nn.Module):
    """Multi-head causal self-attention with dropout, computed for all heads at once.

    Args:
        d_in: Number of features in each input token embedding.
        d_out: Number of output features, split evenly across the heads.
        context_length: Maximum number of tokens in a sequence.
        dropout: Probability of dropping an attention weight during training.
        num_heads: Number of heads; it must divide d_out.
        qkv_bias: Whether each query, key, and value projection includes a bias.
    """

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        # Reduce the projection dim to match desired output dim
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        """Compute context vectors with shape (b, num_tokens, d_out).

        Args:
            x: Input embeddings with shape (b, num_tokens, d_in).

        Returns:
            Context vectors with shape (b, num_tokens, d_out).
        """
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds
        # `context_length`, this will result in errors in the mask creation
        # further below. In practice, this is not a problem since the LLM
        # (chapters 4-7 of the book) ensures that inputs do not exceed
        # `context_length` before reaching this forward method.

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

We create a module with `d_out = 2` and two heads, so each head works with
`head_dim = 1`. These sizes are tiny on purpose, to keep the printed tensors
small; they are not useful in a real model.

In [6]:
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


### Follow the shapes

The next cell repeats the first steps of `forward` for the queries and keys,
and prints the shape after each operation.

In [7]:
with torch.no_grad():
    b, num_tokens, _ = batch.shape

    queries = mha.W_query(batch)
    print("After the projection:     ", tuple(queries.shape))

    queries = queries.view(b, num_tokens, mha.num_heads, mha.head_dim)
    print("After .view (split heads):", tuple(queries.shape))

    queries = queries.transpose(1, 2)
    print("After .transpose(1, 2):   ", tuple(queries.shape))

    keys = mha.W_key(batch).view(b, num_tokens, mha.num_heads, mha.head_dim)
    keys = keys.transpose(1, 2)
    attn_scores = queries @ keys.transpose(2, 3)
    print("Attention scores:         ", tuple(attn_scores.shape))

After the projection:      (2, 6, 2)
After .view (split heads): (2, 6, 2, 1)
After .transpose(1, 2):    (2, 2, 6, 1)
Attention scores:          (2, 2, 6, 6)


## 4. Check: the efficient version equals separate heads

The two implementations compute the same function. To check it, we copy the
weights of `mha` into a wrapper with two heads of size `head_dim`.

`nn.Linear` stores its weight with shape `(d_out, d_in)`: one row per output
feature. Since `.view` assigns the first `head_dim` output features to head 0,
the next ones to head 1, and so on, the projection of head `h` is the block of
rows `h * head_dim` to `(h + 1) * head_dim`.

The wrapper has no output projection, so we apply `mha.out_proj` to its
concatenated output.

In [8]:
head_dim = mha.head_dim
wrapper = MultiHeadAttentionWrapper(
    d_in, head_dim, context_length, 0.0, num_heads=mha.num_heads
)

with torch.no_grad():
    for h, head in enumerate(wrapper.heads):
        rows = slice(h * head_dim, (h + 1) * head_dim)
        head.W_query.weight.copy_(mha.W_query.weight[rows])
        head.W_key.weight.copy_(mha.W_key.weight[rows])
        head.W_value.weight.copy_(mha.W_value.weight[rows])

    wrapper_output = mha.out_proj(wrapper(batch))
    print("Same result:", torch.allclose(wrapper_output, mha(batch)))

Same result: True


## 5. Check: the causal mask still holds

As in the previous notebook, changing the last token of a sequence must not
change the outputs of the tokens before it, in any head.

In [9]:
changed_batch = batch.clone()
changed_batch[1, -1] = torch.tensor([9.0, -9.0, 9.0])

with torch.no_grad():
    original_output = mha(batch)
    changed_output = mha(changed_batch)

print(
    "Earlier tokens unchanged:",
    torch.allclose(original_output[1, :-1], changed_output[1, :-1]),
)
print(
    "Last token changed:",
    not torch.allclose(original_output[1, -1], changed_output[1, -1]),
)

Earlier tokens unchanged: True
Last token changed: True


## 6. How many parameters?

Splitting `d_out` across the heads does not change the size of the query,
key, and value projections: they have the same parameters as a single head
with the same `d_out`. The only addition is the output projection, a
`d_out × d_out` matrix plus a bias of size `d_out`.

In [10]:
def count_parameters(module):
    """Return the number of trainable parameters of a module."""
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


single_head = CausalAttention(d_in, d_out, context_length, 0.0)

print("CausalAttention, one head:", count_parameters(single_head))
print("MultiHeadAttention:       ", count_parameters(mha))
print("Output projection:        ", count_parameters(mha.out_proj))

CausalAttention, one head: 18
MultiHeadAttention:        24
Output projection:         6


The difference, 6 parameters, is exactly the output projection:
`2 × 2` weights plus 2 biases. The mask is a buffer, not a parameter, so it is
not counted.

At the scale of GPT-2 Small (`d_in = d_out = 768`, 12 heads, with biases), the
query, key, and value projections have about 1.77 million parameters and the
output projection about 0.59 million, in each of its 12 transformer blocks.

## Shapes and limits of this example

| Tensor | Shape |
| --- | --- |
| Input batch | `(b, num_tokens, d_in)` |
| Queries, keys, values after the projection | `(b, num_tokens, d_out)` |
| Queries, keys, values split into heads | `(b, num_heads, num_tokens, head_dim)` |
| Attention scores and weights | `(b, num_heads, num_tokens, num_tokens)` |
| Heads combined, and output | `(b, num_tokens, d_out)` |

As in `CausalAttention`, inputs longer than `context_length` are not
supported. The input has no positional information yet: in the GPT model,
positional embeddings are added to the token embeddings before attention.

Multi-head attention is the core of the **transformer block**. The next step
is to build that block around it: layer normalization, a feed-forward
network, and residual connections.